# Phase 4: Explainability (SHAP)

In this notebook, we use SHAP to explain the predictions of our Fake News Detection model.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')

import joblib
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Custom module
from src.explainability import get_shap_explainer, plot_shap_summary, plot_shap_bar, explain_single_prediction

plt.rcParams.update({'figure.dpi': 150})
np.random.seed(42)

## 1. Load Model, Data, and Features

In [ ]:
model_path = '../models/fake_news_model.pkl'
metadata_path = '../models/model_metadata.pkl'
features_path = '../outputs/features.npz'
feature_names_path = '../outputs/feature_names.pkl'

try:
    model = joblib.load(model_path)
    metadata = joblib.load(metadata_path)
    data = np.load(features_path)
    X, y = data['X'], data['y']
    feature_names = joblib.load(feature_names_path)
    print("Successfully loaded model and features!")
except Exception as e:
    print(f"Error loading files: {e}")
    print("Please run 02_features.ipynb and 03_train.ipynb first.")
    # Dummy data to allow notebook to run if files missing
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression().fit(np.random.rand(10, 5), np.random.randint(0, 2, 10))
    X = np.random.rand(100, 5)
    y = np.random.randint(0, 2, 100)
    feature_names = [f'feat_{i}' for i in range(5)]

## 2. Train-Test Split
We need a sample from the training set as background, and samples from the test set to explain.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Sample test instances for explanation
n_samples = 50
sample_idx = np.random.choice(X_test.shape[0], size=n_samples, replace=False)
X_sample = X_test[sample_idx]

## 3. Create SHAP Explainer and Compute Values

In [ ]:
# Use background dataset for initialization
X_background = X_train[np.random.choice(X_train.shape[0], min(500, X_train.shape[0]), replace=False)]
explainer = get_shap_explainer(model, X_background)

print(f"Computing SHAP values for {n_samples} samples...")
shap_values = explainer.shap_values(X_sample)

# Handle multi-class output (taking index 1 for Fake class)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

## 4. SHAP Visualizations
### SHAP Summary Plot

In [ ]:
plot_shap_summary(shap_values, X_sample, feature_names, output_path='../outputs/shap_summary.png')
print("SHAP summary plot saved to ../outputs/shap_summary.png")
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=True)

### SHAP Bar Plot (Global Importance)

In [ ]:
plot_shap_bar(shap_values, feature_names, output_path='../outputs/shap_bar.png')
print("SHAP bar plot saved to ../outputs/shap_bar.png")
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, plot_type='bar', show=True)

## 5. Single Instance Explanation

In [ ]:
idx = 0
print("Explaining a single prediction (Instance 0 in sample):")
res = explain_single_prediction(explainer, X_sample[idx], feature_names)
print("\nTop features pushing toward FAKE:")
for f in res['top_positive']:
    print(f"  {f['feature']}: {f['shap_value']:.4f}")

print("\nTop features pushing toward REAL:")
for f in res['top_negative']:
    print(f"  {f['feature']}: {f['shap_value']:.4f}")